In [ ]:
!pip install segmentation_models_pytorch
!pip install torchinfo

In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image, ImageOps
import os
import numpy as np
from torchvision import transforms
from torch.utils.data import DataLoader
import torchvision.transforms.functional as F
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from torch.utils.data import random_split
import torch.nn as nn
from torchvision import models
from torchvision.models import resnet18, ResNet18_Weights
from torch.cuda.amp import GradScaler
from tqdm import tqdm
import copy
import cv2
import csv
import time
import pandas as pd
from google.colab import drive
from contextlib import nullcontext
import torch
import segmentation_models_pytorch as smp
from torchinfo import summary
from segmentation_models_pytorch import losses

In [ ]:
drive.mount('/content/drive')

In [ ]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
torch.cuda.empty_cache()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## Dataset treatment

In [ ]:
class SegmentationTransform:
    def __init__(self, size=(256, 256)):
        self.size = size

    def __call__(self, image, mask):

        # --- 1. Resize image and mask ---
        image = F.resize(image, self.size)
        mask = F.resize(mask, self.size, interpolation=F.InterpolationMode.NEAREST)

        # --- 2. Convert image to tensor ---
        image = F.to_tensor(image)

        # --- 3. Normalize image ---
        image = F.normalize(
            image,
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )

        # --- 4. Convert mask to multiclass tensor ---
        mask = torch.tensor(np.array(mask), dtype=torch.long)

        return image, mask

In [ ]:
class TACODataset(Dataset):
    def __init__(self, root_data, root_masks, transform=None, num_batches=15):
        self.root_data = root_data
        self.root_masks = root_masks
        self.transform = transform

        self.images = []

        # Now we assume batch_1 to batch_15
        for batch in range(1, num_batches + 1):
            data_dir = os.path.join(root_data, f"batch_{batch}")
            if not os.path.exists(data_dir):
                print(f"Warning: {data_dir} does not exist, skipping.")
                continue

            for fname in sorted(os.listdir(data_dir)):
                if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                    self.images.append((batch, fname))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        batch, fname = self.images[idx]

        img_path = os.path.join(self.root_data, f"batch_{batch}", fname)

        # Convert image name to corresponding PNG
        mask_name = os.path.splitext(fname)[0] + ".png"
        mask_path = os.path.join(self.root_masks, f"batch_{batch}", mask_name)

        # Open RGB image correctly
        image = Image.open(img_path).convert("RGB")
        image = ImageOps.exif_transpose(image)

        # Open mask WITHOUT converting to grayscale
        mask = Image.open(mask_path)   # maintains original mode (usually "P")

        # Apply transformations
        if self.transform:
            image, mask = self.transform(image, mask)

        return image, mask, fname

In [ ]:
transform = SegmentationTransform(size=(256, 256))



root_data = "/content/data" # Have to download the original images from the TACO dataset
root_masks = "/content/masks"

dataset = TACODataset(root_data, root_masks, transform)
train_size = int(0.8 * len(dataset))   # 80% for training
val_size = len(dataset) - train_size    # 20% for validation

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

batch_size = 8

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
print("Number of training images:", len(train_dataset))
print("Number of validation images:", len(val_dataset))

In [ ]:
def denormalize(img):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    return img * std + mean

In [ ]:
def get_custom_cmap():
    colors = [
        (0.0, 0.0, 0.5),   # 0 - Fundo (azul escuro)
        (1.0, 1.0, 0.0),   # 1 - Plástico (amarelo)
        (0.0, 0.8, 0.0),   # 2 - Papel (verde)
        (1.0, 0.0, 0.0),   # 3 - Metal (vermelho)
        (0.6, 0.0, 0.6)    # 4 - Outros (roxo)
    ]

    return ListedColormap(colors)

In [ ]:
images, masks, names = next(iter(train_loader))

for i in range(4):
    img = images[i]
    mask = masks[i]
    name = names[i]

    img = denormalize(img).permute(1,2,0).cpu().numpy()
    img = np.clip(img, 0, 1)

    plt.figure(figsize=(8,4))

    # Original image
    plt.subplot(1,2,1)
    plt.imshow(img)
    plt.title(f"Image: {name}")
    plt.axis("off")

    # Multiclass mask
    plt.subplot(1,2,2)
    cmap = get_custom_cmap()
    plt.imshow(mask.cpu(), cmap=cmap, vmin=0, vmax=4)
    plt.title("Associated mask (multiclass)")
    plt.axis("off")

    plt.show()

In [ ]:
def show_dataset_sample_by_name(dataset, fname):


    # find the image index
    idx = None
    for i, (_, name) in enumerate(dataset.images):
        if name == fname:
            idx = i
            break

    if idx is None:
        print(f"❌ File '{fname}' not found in dataset.")
        return

    # load example (with transformations)
    image, mask, returned_name = dataset[idx]

    # denormalize
    def denormalize(img):
        mean = torch.tensor([0.485, 0.456, 0.406]).reshape(3,1,1)
        std  = torch.tensor([0.229, 0.224, 0.225]).reshape(3,1,1)
        return (img * std + mean).clamp(0,1)

    img_np = denormalize(image).permute(1,2,0).cpu().numpy()

    # multiclass mask is already in [H, W] format
    mask_np = mask.cpu().numpy()

    # use custom colormap
    cmap = get_custom_cmap()

    plt.figure(figsize=(10,4))

    plt.subplot(1,2,1)
    plt.imshow(img_np)
    plt.title(f"Image: {returned_name}")
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.imshow(mask_np, cmap=cmap, vmin=0, vmax=4)
    plt.title("Multiclass Mask")
    plt.axis("off")

    plt.show()

## Model

In [ ]:
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",     # Pre-trained weights
    in_channels=3,                  # RGB
    classes=5                       # Background + 4 categories
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

summary(model, input_size=(batch_size, 3, 256, 256))


## Training

In [ ]:
def multiclass_dice_loss(pred, target, num_classes=5, eps=1e-6):


    pred_soft = F.softmax(pred, dim=1)  # probabilities

    target_onehot = F.one_hot(target, num_classes=num_classes)  # [B, H, W, C]
    target_onehot = target_onehot.permute(0, 3, 1, 2).float()   # [B, C, H, W]

    dims = (0, 2, 3)  # sum over batch and spatial dims

    intersection = torch.sum(pred_soft * target_onehot, dims)
    cardinality  = torch.sum(pred_soft + target_onehot, dims)

    dice = (2. * intersection + eps) / (cardinality + eps)

    return 1 - dice.mean()

In [ ]:
# Colors in BGR (OpenCV) corresponding to the colormap:
# 0: background (not used)
# 1: plastic  -> yellow
# 2: paper     -> green
# 3: metal     -> red
# 4: others    -> purple

CLASS_COLORS_BGR = {
    1: (0, 255, 255),   # yellow  (B,G,R)
    2: (0, 204, 0),     # green
    3: (0, 0, 255),     # red
    4: (153, 0, 153),   # purple
}

# colors in RGB 0-255
CLASS_COLORS_RGB = {
    1: (255, 255, 0),   # yellow
    2: (0, 204, 0),     # green
    3: (255, 0, 0),     # red
    4: (153, 0, 153),   # purple
}

In [ ]:
NUM_CLASSES = 5

def multiclass_iou(logits, targets, num_classes=NUM_CLASSES):

    with torch.no_grad():
        preds = torch.argmax(logits, dim=1)  # [B,H,W]

        ious = []
        for cls in range(num_classes):

            # create binary masks
            pred_mask = (preds == cls)
            true_mask = (targets == cls)

            # intersection and union
            intersection = (pred_mask & true_mask).sum().item()
            union = (pred_mask | true_mask).sum().item()

            if union == 0:
                ious.append(np.nan)  # missing class → do not count
            else:
                ious.append(intersection / union)

        return ious

In [ ]:
def multiclass_dice_score(logits, targets, num_classes=NUM_CLASSES, smooth=1e-6):
    """
    Returns Dice per class as a Python list of floats.
    """

    with torch.no_grad():
        preds = torch.argmax(logits, dim=1)

        dices = []
        for cls in range(num_classes):

            pred_mask = (preds == cls).float()
            true_mask = (targets == cls).float()

            intersection = (pred_mask * true_mask).sum().item()
            total = pred_mask.sum().item() + true_mask.sum().item()

            if total == 0:
                dices.append(np.nan)
            else:
                dices.append((2 * intersection + smooth) / (total + smooth))

        return dices

In [ ]:
def to_float(x):
    """Converts tensor, numpy or float to a safe float."""
    if isinstance(x, torch.Tensor):
        return float(x.detach().cpu().item())
    try:
        return float(x)
    except:
        return np.nan

In [ ]:
def mean_without_background(values):
    """values = list with IoU/Dice for classes [0..4].
       Returns mean of only classes 1..4."""
    fg = [v for i, v in enumerate(values) if i != 0]
    return float(np.nanmean(fg))

In [ ]:
def overlay_contours_on_image(base_img, mask_np, color_rgb=(255,255,0), thickness=1):
    """
    base_img: numpy RGB [H,W,3] uint8
    color_rgb: tuple (R,G,B)
    """
    # ensure types
    img_rgb = base_img.copy()
    mask_uint8 = (mask_np * 255).astype(np.uint8) if mask_np.max() <= 1 else mask_np.astype(np.uint8)

    # convert image to BGR for OpenCV
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)

    contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # convert color from RGB -> BGR
    color_bgr = (color_rgb[2], color_rgb[1], color_rgb[0])

    cv2.drawContours(img_bgr, contours, -1, color_bgr, thickness)

    # convert back to RGB for matplotlib
    img_rgb_out = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return img_rgb_out

In [ ]:
def overlay_multiclass_contours(base_img, mask_np, class_colors=CLASS_COLORS_RGB, thickness=1):
    """
    base_img: RGB uint8
    mask_np:  [H,W] with values 0..4
    """
    img_rgb = base_img.copy()
    mask_int = mask_np.astype(np.uint8)
    mask_int[mask_int > 4] = 0  # safety

    for cls_id, color_rgb in class_colors.items():
        class_mask = (mask_int == cls_id).astype(np.uint8)
        if class_mask.max() == 0:
            continue

        img_rgb = overlay_contours_on_image(img_rgb, class_mask, color_rgb=color_rgb, thickness=thickness)

    return img_rgb

In [ ]:
def show_predictions(model, loader, device, max_images=4):
    model.eval()
    images, masks, names = next(iter(loader))
    images = images.to(device)
    masks  = masks.to(device)

    with torch.no_grad():
        logits = model(images)
        preds  = torch.argmax(logits, dim=1)

    images = images.cpu()
    masks  = masks.cpu()
    preds  = preds.cpu()

    max_images = min(max_images, images.size(0))

    for i in range(max_images):
        img = denormalize(images[i]).permute(1,2,0).numpy()
        img = np.clip(img, 0, 1)
        img_255 = (img * 255).astype(np.uint8)

        mask_real = masks[i].numpy()
        mask_pred = preds[i].numpy()

        real_overlay = overlay_multiclass_contours(img_255, mask_real, CLASS_COLORS_RGB, thickness=1)
        pred_overlay = overlay_multiclass_contours(img_255, mask_pred, CLASS_COLORS_RGB, thickness=1)

        plt.figure(figsize=(15,5))
        plt.subplot(1,3,1); plt.imshow(img_255);      plt.title(f"Image\n{names[i]}"); plt.axis("off")
        plt.subplot(1,3,2); plt.imshow(real_overlay); plt.title("Real mask (contours, classes)"); plt.axis("off")
        plt.subplot(1,3,3); plt.imshow(pred_overlay); plt.title("Predicted mask (contours, classes)"); plt.axis("off")
        plt.show()

In [ ]:
def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    num_epochs=20,
    device="cuda",
    save_every=5,
    checkpoint_path="/content/drive/MyDrive/taco_resume_classes.pth"
):

     # ======== MULTICLASS LOSS WITH WEIGHTS =========
    class_weights = torch.tensor([1.0, 10.0, 10.0, 10.0, 10.0]).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    # ======== AMP =========
    use_amp = hasattr(torch, "amp") and hasattr(torch.amp, "autocast")

    if use_amp and hasattr(torch.amp, "GradScaler"):
        scaler = torch.amp.GradScaler("cuda")
    else:
        from torch.cuda.amp import GradScaler
        scaler = GradScaler()

    # ======== AUTO-RESUME =========
    model, optimizer, scaler, history, best_epoch, completed_epochs = \
        load_checkpoint_if_exists(model, optimizer, scaler, checkpoint_path, device=device)

    if history is None:
        history = {
            "epoch": [], "train_loss": [], "train_iou": [], "train_dice": [],
            "val_loss": [], "val_iou": [], "val_dice": [], "time_sec": []
        }
        best_loss = float("inf")
        best_epoch = 0
    else:
        best_loss = history["val_loss"][best_epoch - 1]

    print(f"➡ Resuming training from epoch {completed_epochs + 1}")

    def get_amp_ctx():
        return torch.amp.autocast(device_type="cuda", dtype=torch.float16) if use_amp else nullcontext()


    # =============== TRAINING ===============
    for epoch in range(completed_epochs, num_epochs):
        print(f"\n🔵 Epoch {epoch+1}/{num_epochs}")
        print("-" * 40)

        t0 = time.time()
        model.train()
        train_loss = train_iou = train_dice = 0.0

        for images, masks, _ in tqdm(train_loader, desc="Training"):
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad()

            with get_amp_ctx():
                outputs = model(images)
                loss = criterion(outputs, masks)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            # ====== METRICS ======
            ious = multiclass_iou(outputs, masks)
            dices = multiclass_dice_score(outputs, masks)

            ious_f  = [to_float(x) for x in ious]
            dices_f = [to_float(x) for x in dices]

            mean_iou  = np.nanmean(ious_f[1:])   # ignores background
            mean_dice = np.nanmean(dices_f[1:])  # ignores background

            bs = images.size(0)
            train_loss += loss.item() * bs
            train_iou  += mean_iou * bs
            train_dice += mean_dice * bs


        train_loss /= len(train_loader.dataset)
        train_iou  /= len(train_loader.dataset)
        train_dice /= len(train_loader.dataset)


        # =============== VALIDATION ===============
        model.eval()
        val_loss = val_iou = val_dice = 0.0

        with torch.no_grad():
            for images, masks, _ in val_loader:
                images, masks = images.to(device), masks.to(device)

                with get_amp_ctx():
                    outputs = model(images)
                    loss = criterion(outputs, masks)

                ious = multiclass_iou(outputs, masks)
                dices = multiclass_dice_score(outputs, masks)

                ious_f  = [to_float(x) for x in ious]
                dices_f = [to_float(x) for x in dices]

                mean_iou  = np.nanmean(ious_f[1:])
                mean_dice = np.nanmean(dices_f[1:])

                bs = images.size(0)
                val_loss += loss.item() * bs
                val_iou  += mean_iou * bs
                val_dice += mean_dice * bs

        val_loss /= len(val_loader.dataset)
        val_iou  /= len(val_loader.dataset)
        val_dice /= len(val_loader.dataset)

        # ===== HISTORY =====
        history["epoch"].append(epoch+1)
        history["train_loss"].append(train_loss)
        history["train_iou"].append(train_iou)
        history["train_dice"].append(train_dice)
        history["val_loss"].append(val_loss)
        history["val_iou"].append(val_iou)
        history["val_dice"].append(val_dice)
        history["time_sec"].append(time.time() - t0)

        print(f"Train Loss={train_loss:.4f} | IoU={train_iou:.4f} | Dice={train_dice:.4f}")
        print(f"Val   Loss={val_loss:.4f} | IoU={val_iou:.4f} | Dice={val_dice:.4f}")

        # ===== BEST MODEL =====
        if val_loss < best_loss:
            best_loss = val_loss
            best_epoch = epoch + 1
            best_weights = copy.deepcopy(model.state_dict())

        if (epoch + 1) % save_every == 0:
            save_checkpoint(model, optimizer, scaler, history, best_epoch, best_loss, epoch + 1, checkpoint_path)

        show_predictions(model, val_loader, device, max_images=6)

    print(f"\n🏆 Best Val Loss: {best_loss:.4f} (epoch {best_epoch})")
    model.load_state_dict(best_weights)

    return model, history, best_epoch, best_loss

In [ ]:
def save_checkpoint(model, optimizer, scaler, history, best_epoch, best_loss, completed_epochs, path):
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "history": history,
        "best_epoch": best_epoch,
        "best_loss": best_loss,
        "completed_epochs": completed_epochs
    }
    torch.save(checkpoint, path)
    print(f"💾 Checkpoint saved to {path}")

def load_checkpoint_if_exists(model, optimizer, scaler, path, device="cuda"):
    if not os.path.exists(path):
        print("No checkpoint found. Training from scratch.")
        return model, optimizer, scaler, None, None, 0  # no resume

    print(f"🔄 Loading checkpoint: {path}")

    # Add safe globals for numpy before loading
    torch.serialization.add_safe_globals([np._core.multiarray.scalar])

    checkpoint = torch.load(path, map_location=device, weights_only=False)

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])

    history = checkpoint["history"]
    best_epoch = checkpoint["best_epoch"]
    best_loss = checkpoint["best_loss"]
    completed_epochs = checkpoint["completed_epochs"]

    print(f"✔ Checkpoint restored. Last epoch completed: {completed_epochs}")
    print(f"✔ Previous Best Val Loss: {best_loss:.4f}")

    return model, optimizer, scaler, history, best_epoch, completed_epochs

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

model, history, best_epoch, best_loss = train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    num_epochs=50,
    device=device,
    save_every=5,
    checkpoint_path="/content/drive/MyDrive/taco_resume_classes.pth"
)

## Salvar modelo

In [ ]:
def save_checkpoint(model, history, best_epoch, best_loss, optimizer=None, path="/content/drive/MyDrive/taco_best_checkpoint_classes.pth"): # path="/content/drive/Othercomputers/Meu laptop/EI/EI7/Introduction_IA/Projet/modelos/taco_best_checkpoint_classes.pth"):
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "history": history,
        "best_epoch": best_epoch,
        "best_loss": best_loss,
    }
    if optimizer:
        checkpoint["optimizer_state_dict"] = optimizer.state_dict()

    torch.save(checkpoint, path)
    print("💾 Checkpoint saved to:", path)

In [ ]:
def save_history_csv(history, path="/content/drive/MyDrive/taco_history_classes.csv"):
    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(history.keys())
        writer.writerows(zip(*history.values()))
    print("📊 History saved to:", path)

In [ ]:
save_checkpoint(model, history, best_epoch, best_loss, optimizer)
save_history_csv(history)

## Carregar modelo


In [ ]:
checkpoint = torch.load("/content/drive/MyDrive/taco_resume_classes.pth", map_location="cuda", weights_only=False)


model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

history = checkpoint["history"]
best_epoch = checkpoint["best_epoch"]
best_loss = checkpoint["best_loss"]

## Plotar dados treinamento

In [ ]:
def plot_training_history(history):
    """
    history can be:
       - dict returned by train_model
       - or a DataFrame loaded from taco_history.csv
    """

    if isinstance(history, dict):
        df = pd.DataFrame(history)
    else:
        df = history.copy()

    plt.figure(figsize=(14, 5))

    # ---------- LOSS ----------
    plt.subplot(1, 3, 1)
    plt.plot(df["epoch"], df["train_loss"], label="Train Loss")
    plt.plot(df["epoch"], df["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Loss Curve")
    plt.grid(True)
    plt.legend()

    # ---------- IOU ----------
    plt.subplot(1, 3, 2)
    plt.plot(df["epoch"], df["train_iou"], label="Train IoU")
    plt.plot(df["epoch"], df["val_iou"], label="Val IoU")
    plt.xlabel("Epoch")
    plt.ylabel("IoU")
    plt.title("IoU Curve")
    plt.grid(True)
    plt.legend()

    # ---------- DICE ----------
    plt.subplot(1, 3, 3)
    plt.plot(df["epoch"], df["train_dice"], label="Train Dice")
    plt.plot(df["epoch"], df["val_dice"], label="Val Dice")
    plt.xlabel("Epoch")
    plt.ylabel("Dice")
    plt.title("Dice Curve")
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
history_df = pd.read_csv("/content/drive/MyDrive/taco_history_classes.csv")
plot_training_history(history_df)

## Model Test

In [ ]:
def evaluate_model_on_test(model, test_loader, device="cuda"):
    model.eval()

    class_weights = torch.tensor([0.1, 1.0, 1.0, 1.0, 1.0]).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    total_loss = 0
    total_iou = 0
    total_dice = 0
    total_samples = 0

    with torch.no_grad():
        for images, masks, _ in test_loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)


            loss = criterion(outputs, masks)

            ious  = multiclass_iou(outputs, masks)
            dices = multiclass_dice_score(outputs, masks)


            ious_f  = [float(i.detach().cpu()) if isinstance(i, torch.Tensor) else float(i) for i in ious]
            dices_f = [float(d.detach().cpu()) if isinstance(d, torch.Tensor) else float(d) for d in dices]


            mean_iou  = mean_without_background(ious_f)
            mean_dice = mean_without_background(dices_f)

            bs = images.size(0)

            total_samples += bs
            total_loss += loss.item() * bs
            total_iou  += mean_iou * bs
            total_dice += mean_dice * bs

    metrics = {
        "test_loss": total_loss / total_samples,
        "test_iou":  total_iou / total_samples,
        "test_dice": total_dice / total_samples
    }

    return metrics

In [ ]:
def plot_test_results(metrics):
    plt.figure(figsize=(7,5))

    names = ["Test Loss", "IoU (no background)", "Dice (no background)"]
    values = [metrics["test_loss"], metrics["test_iou"], metrics["test_dice"]]

    colors = ["#d62728", "#1f77b4", "#2ca02c"]  # red, blue, green

    bars = plt.bar(names, values, color=colors)

    # Show numeric value above each bar
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2.,
                 height + 0.02,
                 f"{height:.4f}",
                 ha="center", fontsize=12)

    plt.title("Multiclass Segmentation – Test Performance", fontsize=14)
    plt.ylim(0, 1.15)
    plt.ylabel("Score", fontsize=12)
    plt.grid(axis="y", linestyle="--", alpha=0.4)

    plt.show()

In [ ]:
metrics = evaluate_model_on_test(model, val_loader, device)
print(metrics)
plot_test_results(metrics)
show_predictions(model, val_loader, device, max_images=10)

## Exportar Grafo Executável

In [ ]:
model.eval()

dummy_input = torch.randn(1, 3, 256, 256).to(device)

try:
    print("🔍 Attempting TorchScript via script()...")
    ts_model = torch.jit.script(model)
except Exception as e:
    print("⚠️ script() failed, using trace()")
    ts_model = torch.jit.trace(model, dummy_input)

#export_path = "/content/drive/Othercomputers/Meu laptop/EI/EI7/Introduction_IA/Projet/modelos/taco_unet_model_classes.pt"
export_path = "/content/drive/MyDrive/taco_unet_model_classes.pt"
torch.jit.save(ts_model, export_path)

print("✅ TorchScript model exported to:", export_path)